# Meta-Ensemble: RF + ANN + Monte Carlo Equity

This notebook builds a **stacking ensemble** that combines three different signals about a poker hand:

1. **Random Forest** — predicted probability distribution over the 10 final-hand-strength classes
2. **ANN** — same, from the neural net
3. **Monte Carlo equity** — win/tie/lose probability vs. a random opponent, estimated by dealing out the remaining deck thousands of times and evaluating actual poker hands (a completely different kind of signal: head-to-head equity rather than "what category will this hand be")

Three meta-learners (Logistic Regression, Linear Regression, and a small ANN) are trained on these 23 combined features (`10 RF probs + 10 ANN probs + 3 MC probs`) to predict the true final hand class, and compared against RF/ANN alone on the same held-out hands

In [ ]:
import numpy as np, pandas as pd, random, time, json
from collections import Counter
from itertools import combinations
import joblib

## 1. Poker hand evaluator (Python port of the Monte Carlo logic)



In [ ]:
def hand_rank_5(cards):
    """cards: list of 5 (suit, rank) tuples. Returns a comparable tuple, higher = better hand."""
    ranks = sorted([r for s, r in cards], reverse=True)
    suits = [s for s, r in cards]
    rank_counts = Counter(ranks)
    counts = sorted(rank_counts.items(), key=lambda x: (-x[1], -x[0]))
    count_vals = [c for r, c in counts]
    is_flush = len(set(suits)) == 1
    unique_ranks = sorted(set(ranks), reverse=True)
    is_straight = False; straight_high = None
    if len(unique_ranks) == 5:
        if unique_ranks[0] - unique_ranks[4] == 4:
            is_straight = True; straight_high = unique_ranks[0]
        elif unique_ranks == [14, 5, 4, 3, 2]:            # wheel: A-2-3-4-5
            is_straight = True; straight_high = 5
    if is_straight and is_flush:
        return (8, straight_high)                          # straight flush / royal flush
    elif count_vals[0] == 4:
        return (7, counts[0][0], counts[1][0])              # four of a kind
    elif count_vals[0] == 3 and count_vals[1] == 2:
        return (6, counts[0][0], counts[1][0])              # full house
    elif is_flush:
        return (5,) + tuple(sorted(ranks, reverse=True))    # flush
    elif is_straight:
        return (4, straight_high)                           # straight
    elif count_vals[0] == 3:
        kickers = sorted([r for r, c in counts if c == 1], reverse=True)
        return (3, counts[0][0]) + tuple(kickers)           # three of a kind
    elif count_vals[0] == 2 and count_vals[1] == 2:
        pairs = sorted([r for r, c in counts if c == 2], reverse=True)
        kicker = [r for r, c in counts if c == 1]
        return (2,) + tuple(pairs) + tuple(kicker)          # two pair
    elif count_vals[0] == 2:
        kickers = sorted([r for r, c in counts if c == 1], reverse=True)
        return (1, counts[0][0]) + tuple(kickers)           # one pair
    else:
        return (0,) + tuple(sorted(ranks, reverse=True))    # high card

def best_of_seven(cards7):
    best = None
    for combo in combinations(cards7, 5):
        r = hand_rank_5(list(combo))
        if best is None or r > best:
            best = r
    return best

In [ ]:

SUIT_MAP = {'♠':0,'♥':1,'♦':2,'♣':3}
RANK_MAP = {'A':14,'K':13,'Q':12,'J':11,'10':10,'9':9,'8':8,'7':7,'6':6,'5':5,'4':4,'3':3,'2':2}
def c(s): return (SUIT_MAP[s[0]], RANK_MAP[s[1:]])

assert hand_rank_5([c('♠A'),c('♠K'),c('♠Q'),c('♠J'),c('♠' + '10')]) > \
       hand_rank_5([c('♥9'),c('♥8'),c('♥7'),c('♥6'),c('♥5')])          # royal beats straight flush
assert hand_rank_5([c('♠7'),c('♥7'),c('♦7'),c('♣7'),c('♠2')]) > \
       hand_rank_5([c('♠K'),c('♥K'),c('♦K'),c('♣2'),c('♠2')])          # quads beat full house
assert hand_rank_5([c('♠A'),c('♥2'),c('♦3'),c('♣4'),c('♠5')])[0] == 4  # wheel counts as a straight
print('All hand ranking checks passed')

All hand ranking checks passed


## 2. Monte Carlo equity estimator

Same algorithm as `montecarlo.cpp`: build the deck of unseen cards, shuffle, deal a random opponent hand plus however many board cards are still missing, evaluate both 7-card hands, and tally win/tie/loss over many trials.

In [ ]:
def build_remaining_deck(known):
    known_set = set(known)
    return [(s, r) for s in range(4) for r in range(2, 15) if (s, r) not in known_set]

def estimate_equity(hero_hole, board, num_sims, rng):
    known = hero_hole + board
    deck = build_remaining_deck(known)
    cards_needed = 5 - len(board)
    wins = ties = losses = 0
    for _ in range(num_sims):
        rng.shuffle(deck)
        opp = deck[0:2]
        sim_board = board + deck[2:2 + cards_needed]
        hero7 = hero_hole + sim_board
        opp7 = opp + sim_board
        hr = best_of_seven(hero7)
        orr = best_of_seven(opp7)
        if hr > orr: wins += 1
        elif orr > hr: losses += 1
        else: ties += 1
    return wins / num_sims, ties / num_sims, losses / num_sims

rng = random.Random(42)
w, t, l = estimate_equity([c('♠A'), c('♥A')], [], 200, rng)
print(f"Pocket Aces preflop equity vs random hand: win={w:.1%} tie={t:.1%} lose={l:.1%}")

Pocket Aces preflop equity vs random hand: win=86.0% tie=0.5% lose=13.5%


86% matches the commonly-quoted preflop equity for pocket Aces vs. a random hand, which is a good sanity check that the simulator is behaving correctly.

## 3. Build the meta-dataset

To keep this a fair, uncontaminated test of stacking, the meta-dataset uses **2,000 hands that were never used to train RF or ANN** — they're drawn from the 900,000 hands *not* included in the 100,000-hand sample used earlier. For each of the resulting 6,000 stage-rows (Preflop/Flop/Turn × 2,000 hands), we compute Monte Carlo equity (200 trials each) plus RF's and ANN's predicted probability distributions.

In [ ]:
n_hands_total = 1_000_000
RANDOM_STATE = 42
SAMPLE_HANDS = 100000
rng0 = np.random.RandomState(RANDOM_STATE)
base_hand_ids = rng0.choice(np.arange(n_hands_total), size=SAMPLE_HANDS, replace=False)
base_hand_set = set(base_hand_ids.tolist())    # hands already used to train RF/ANN

full_transformed = pd.read_pickle("full_transformed.pkl")   

META_N_HANDS = 2000
available = np.array([h for h in np.arange(n_hands_total) if h not in base_hand_set][:META_N_HANDS*3])
rng1 = np.random.RandomState(123)
meta_hand_ids = rng1.choice(available, size=META_N_HANDS, replace=False)

idx = np.concatenate([np.array([3*h, 3*h+1, 3*h+2]) for h in meta_hand_ids])
meta_rows = full_transformed.iloc[idx].reset_index(drop=True)
print("meta_rows:", meta_rows.shape)

meta_rows: (6000, 34)


In [ ]:
STAGE_N_COMMUNITY = {0: 0, 1: 3, 2: 4}   # Preflop/Flop/Turn -> how many community cards are known
NUM_SIMS = 200
rng = random.Random(42)

win_l, tie_l, lose_l = [], [], []
for i, row in meta_rows.iterrows():
    hero = [(int(row['c1_suit']), int(row['c1_rank'])), (int(row['c2_suit']), int(row['c2_rank']))]
    n_comm = STAGE_N_COMMUNITY[int(row['stage'])]
    board = [(int(row[f'c{k}_suit']), int(row[f'c{k}_rank'])) for k in range(3, 3+n_comm)]
    w, t, l = estimate_equity(hero, board, NUM_SIMS, rng)
    win_l.append(w); tie_l.append(t); lose_l.append(l)

meta_rows['mc_win'] = win_l; meta_rows['mc_tie'] = tie_l; meta_rows['mc_lose'] = lose_l
print("Monte Carlo equity computed for all", len(meta_rows), "rows (", NUM_SIMS, "trials each )")

Monte Carlo equity computed for all 6000 rows ( 200 trials each )


## 4. Get RF and ANN predictions, assemble meta-features

23 meta-features per row: `10 RF class-probabilities + 10 ANN class-probabilities + mc_win + mc_tie + mc_lose`.

In [ ]:
from tensorflow.keras.models import load_model

X_train_cols = list(pd.read_pickle("X_train.pkl").columns)
suit_cols = [f'c{i}_suit' for i in range(1, 8)]

base_cols = [col for col in meta_rows.columns if col not in ['mc_win','mc_tie','mc_lose','target']]
base_oh = pd.get_dummies(meta_rows[base_cols], columns=suit_cols+['stage'], prefix=suit_cols+['stage'])
base_oh = base_oh.reindex(columns=X_train_cols, fill_value=0)

best_rf = joblib.load("random_forest_model.joblib")
rf_probs = best_rf.predict_proba(base_oh)

scaler = joblib.load("scaler.joblib")
ann = load_model("ann_model.h5")
ann_probs = ann.predict(scaler.transform(base_oh.astype(np.float32)), verbose=0, batch_size=512)

label_encoder = joblib.load("label_encoder.joblib")
y_meta = label_encoder.transform(meta_rows['target'])

rf_cols = [f'rf_p{cl}' for cl in label_encoder.classes_]
ann_cols = [f'ann_p{cl}' for cl in label_encoder.classes_]
meta_X = pd.DataFrame(rf_probs, columns=rf_cols)
for i, colname in enumerate(ann_cols):
    meta_X[colname] = ann_probs[:, i]
meta_X['mc_win']  = meta_rows['mc_win'].to_numpy()
meta_X['mc_tie']  = meta_rows['mc_tie'].to_numpy()
meta_X['mc_lose'] = meta_rows['mc_lose'].to_numpy()
print("meta_X:", meta_X.shape)

meta_X: (6000, 23)


## 5. Train/test split and the three meta-learners

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score

Xtr, Xte, ytr, yte = train_test_split(meta_X, y_meta, test_size=0.3, random_state=42, stratify=y_meta)
print("meta-train:", Xtr.shape, "meta-test:", Xte.shape)

meta-train: (4200, 23) meta-test: (1800, 23)


**Meta-model 1: Logistic Regression** — the standard, principled choice for a multi-class stacking meta-learner (softmax over classes).

In [ ]:
logreg = LogisticRegression(max_iter=2000)
logreg.fit(Xtr, ytr)
logreg_acc = accuracy_score(yte, logreg.predict(Xte))
print("Logistic Regression meta-model accuracy:", logreg_acc)

Logistic Regression meta-model accuracy: 0.6311111111111111


**Meta-model 2: Linear Regression** — not the standard tool for classification (it doesn't model class probabilities properly and can predict outside [0,1]), but included as requested. Implemented as a "linear probability model": regress each class's one-hot indicator, then take the `argmax` of the predicted scores.

In [ ]:
n_classes = len(label_encoder.classes_)
ytr_onehot = np.eye(n_classes)[ytr]
linreg = LinearRegression()
linreg.fit(Xtr, ytr_onehot)
linreg_pred = linreg.predict(Xte).argmax(axis=1)
linreg_acc = accuracy_score(yte, linreg_pred)
print("Linear Regression meta-model accuracy:", linreg_acc)

Linear Regression meta-model accuracy: 0.6416666666666667


**Meta-model 3: small ANN** — a nonlinear combiner in case the relationship between the three signals isn't linear.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

ytr_cat = to_categorical(ytr, num_classes=n_classes)
yte_cat = to_categorical(yte, num_classes=n_classes)

meta_ann = models.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(n_classes, activation='softmax')
])
meta_ann.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                                               restore_best_weights=True, mode='max')

history = meta_ann.fit(Xtr.to_numpy(), ytr_cat, validation_split=0.15, epochs=60,
                        batch_size=64, callbacks=[early_stop], verbose=2)

_, meta_ann_acc = meta_ann.evaluate(Xte.to_numpy(), yte_cat, verbose=0)
print("Meta ANN accuracy:", meta_ann_acc)

Epoch 1/60
56/56 - 1s - 18ms/step - accuracy: 0.4434 - loss: 1.8927 - val_accuracy: 0.5556 - val_loss: 1.4157
Epoch 5/60
56/56 - 0s - 3ms/step - accuracy: 0.6048 - loss: 0.9417 - val_accuracy: 0.6302 - val_loss: 0.8654
Epoch 10/60
56/56 - 0s - 3ms/step - accuracy: 0.6168 - loss: 0.8981 - val_accuracy: 0.6190 - val_loss: 0.8285
Epoch 15/60
56/56 - 0s - 3ms/step - accuracy: 0.6171 - loss: 0.8782 - val_accuracy: 0.6397 - val_loss: 0.8147
Epoch 20/60
56/56 - 0s - 2ms/step - accuracy: 0.6266 - loss: 0.8662 - val_accuracy: 0.6206 - val_loss: 0.8114
Epoch 25/60
56/56 - 0s - 3ms/step - accuracy: 0.6305 - loss: 0.8559 - val_accuracy: 0.6397 - val_loss: 0.7991
Epoch 30/60
56/56 - 0s - 3ms/step - accuracy: 0.6283 - loss: 0.8477 - val_accuracy: 0.6524 - val_loss: 0.7931
Epoch 35/60
56/56 - 0s - 3ms/step - accuracy: 0.6311 - loss: 0.8459 - val_accuracy: 0.6492 - val_loss: 0.7888
Epoch 37/60
56/56 - 0s - 3ms/step - accuracy: 0.6367 - loss: 0.8440 - val_accuracy: 0.6397 - val_loss: 0.7898
(early stop

## 6. Fair comparison: RF alone and ANN alone on the *same* test rows

In [ ]:
rf_pred_alone  = Xte[rf_cols].to_numpy().argmax(axis=1)
ann_pred_alone = Xte[ann_cols].to_numpy().argmax(axis=1)
rf_acc_alone  = accuracy_score(yte, rf_pred_alone)
ann_acc_alone = accuracy_score(yte, ann_pred_alone)

print(f"{'Model':30s}{'Accuracy':>10s}")
print(f"{'Random Forest alone':30s}{rf_acc_alone:>9.2%}")
print(f"{'ANN alone':30s}{ann_acc_alone:>9.2%}")
print(f"{'Meta: Logistic Regression':30s}{logreg_acc:>9.2%}")
print(f"{'Meta: Linear Regression':30s}{linreg_acc:>9.2%}")
print(f"{'Meta: ANN':30s}{meta_ann_acc:>9.2%}")

Model                           Accuracy
Random Forest alone               63.67%
ANN alone                         64.11%
Meta: Logistic Regression         63.11%
Meta: Linear Regression           64.17%
Meta: ANN                         63.11%


## 7. Honest conclusion

**Stacking did not meaningfully beat the ANN alone.** All five numbers sit within about 1 percentage point of each other, and with only 1,800 held-out test rows, the noise floor on accuracy is roughly ±2% — so none of these differences are statistically distinguishable. The Linear Regression meta-model nominally edges out ANN alone (64.17% vs 64.11%), but that's well inside the noise.

**Why stacking didn't help here, most likely:**
- RF and ANN were trained on the **same features**, so their predictions are highly correlated — there's little complementary signal for a meta-learner to exploit between them.
- Monte Carlo equity answers a genuinely different question (**win probability vs. a random opponent**) than the classification target (**which of 10 categories the final hand lands in**). A hand can have high win equity and still resolve to a common category like PAIR (e.g. top pair vs. random hand), so `mc_win/tie/lose` isn't a strong direct predictor of hand *category* — even though it's a genuinely useful signal for a different question ("should I call this bet").
- 4,200 meta-training rows is a small dataset for a meta-learner to find a subtle nonlinear combination in.

**Where this ensemble likely *would* help:** if the target were **win/loss against an opponent** (what Monte Carlo is actually built to estimate) rather than hand category, or if RF and ANN were trained on genuinely different feature subsets (encouraging them to make different kinds of mistakes for the meta-learner to correct).

## 8. Save the meta-models

In [ ]:
joblib.dump(logreg, 'meta_logistic_regression.joblib')
joblib.dump(linreg, 'meta_linear_regression.joblib')
meta_ann.save('meta_ann.h5')
print("Saved: meta_logistic_regression.joblib, meta_linear_regression.joblib, meta_ann.h5")

Saved: meta_logistic_regression.joblib, meta_linear_regression.joblib, meta_ann.h5
